# NS-RAG pipeline -- synthetic-data pilot run (Synthea + Flan-T5-Large)

This notebook runs the **real** NS-RAG pipeline end-to-end on **fully synthetic** patient data generated by [Synthea](https://github.com/synthetichealth/synthea) -- no real patients, no data use agreement, no credentialing needed.

**What this is:** a wiring/scale validation pilot that produces real (not fabricated) numbers, usable while MIMIC-III/i2b2 credentialing is pending.

**What this is NOT:** a substitute for credentialed MIMIC-III/i2b2 results. Section 5.2 of the paper names those datasets specifically; numbers from this notebook should only appear in the paper if explicitly reported as a synthetic-data robustness/pilot check, clearly labeled as such.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine, free tier).

## 1. Install Java + build Synthea, generate synthetic patients

In [ ]:
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!git clone -q https://github.com/synthetichealth/synthea.git
%cd synthea
# Enable CSV export
!sed -i 's/exporter.csv.export = false/exporter.csv.export = true/' src/main/resources/synthea.properties || echo 'exporter.csv.export = true' >> src/main/resources/synthea.properties
# Generate 50 synthetic patients (a couple of minutes)
!./run_synthea -p 50
%cd ..

## 2. Get the NS-RAG pipeline code

In [ ]:
!git clone -q https://github.com/ANONYMOUS/REPO.git nsrag-pipeline
%cd nsrag-pipeline
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, 'src')

## 3. Build QA instances from the Synthea output

In [ ]:
from nsrag.data.synthea_loader import SyntheaLoader

loader = SyntheaLoader('../synthea/output/csv')
instances = loader.to_qa_instances()
print(f'{len(instances)} QA instances built from synthetic patients')
print('Example question:', instances[0].question)
print('Example note:', instances[0].notes[0].text[:200])

## 4. Run the real pipeline (Flan-T5-Large on GPU)
This uses `HFGenerator`, not `MockGenerator` -- real model, real outputs.

In [ ]:
from nsrag.generation.generator import HFGenerator
from nsrag.baselines.llm_only import LLMOnly
from nsrag.baselines.textual_rag import TextualRAG
from nsrag.baselines.graph_linearization import GraphLinearization
from nsrag.baselines.ns_rag import NSRAGWithoutVerification, NSRAG
from nsrag.pipeline.run_qa import run_qa_experiment

generator = HFGenerator(model_name='google/flan-t5-large', device='cuda', temperature=0.2)

methods = [
    LLMOnly(generator),
    TextualRAG(generator),
    GraphLinearization(generator),
    NSRAGWithoutVerification(generator),
    NSRAG(generator),
]

results = run_qa_experiment(methods, instances)

## 5. Ablation study (same synthetic instances)

In [ ]:
from nsrag.pipeline.run_ablation import run_ablation_experiment

ablation_results = run_ablation_experiment(generator, instances)

## 6. Efficiency (stage-level timing, real GPU numbers)

In [ ]:
from nsrag.pipeline.run_efficiency import run_efficiency_experiment

efficiency = run_efficiency_experiment(generator, instances)
efficiency

## 7. Results as LaTeX rows (synthetic pilot -- NOT credentialed MIMIC-III/i2b2 results)

In [ ]:
from nsrag.utils.results_to_latex import qa_results_to_latex, ablation_results_to_latex, efficiency_results_to_latex

row_order = ['B1 - LLM', 'B2 - Textual RAG', 'B3 - Graph Linearization',
             'B7 - NS-RAG w/o Verification', 'B8 - NS-RAG']
print('--- Synthetic-data QA results (pilot only) ---')
print(qa_results_to_latex(results, row_order))
print()
print('--- Synthetic-data ablation (pilot only) ---')
ablation_row_order = [k for k in ablation_results.keys() if k != '_efficiency']
print(ablation_results_to_latex(ablation_results, ablation_row_order))
print()
print('--- Efficiency (real GPU timing) ---')
print(efficiency_results_to_latex(efficiency))

## 8. (Optional) Save results to your Google Drive

In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')

out = {
    'qa': {k: (v if k == '_efficiency' else {
        'fact_f1': v.fact_f1, 'cr': v.cr, 'ur': v.ur, 'ec': v.ec}) for k, v in results.items()},
    'ablation': {k: (v if k == '_efficiency' else {
        'fact_f1': v.fact_f1, 'cr': v.cr, 'ur': v.ur, 'ec': v.ec}) for k, v in ablation_results.items()},
    'efficiency': efficiency,
}
with open('/content/drive/MyDrive/nsrag_synthetic_pilot_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
print('Saved to Google Drive: nsrag_synthetic_pilot_results.json')